In [299]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
sns.set_theme(palette='pastel')

In [300]:
df_2020 = pd.read_json('../data/data_as_json/meldungen_2020.json')
df_2021 = pd.read_json('../data/data_as_json/meldungen_2021.json')
df_2022 = pd.read_json('../data/data_as_json/meldungen_2022.json')
df_2023 = pd.read_json('../data/data_as_json/meldungen_2023.json')
df_2024 = pd.read_json('../data/data_as_json/meldungen_2024.json')
df_2025 = pd.read_json('../data/data_as_json/meldungen_2025.json')

In [301]:
df = pd.concat([df_2020, df_2021, df_2022, df_2023, df_2024, df_2025])
df.head()

,date,title,location,link,details,number
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316
3,2021-02-04 11:00:00,Gefährliche Körperverletzung,Mitte,/polizei/polizeimeldungen/pressemitteilung.960...,Die Kriminalpolizei der Direktion 2 bittet um ...,0276
4,2021-01-12 13:02:00,Verkehrsunfall mit schwerverletztem E-Bike-Fahrer,Mitte,/polizei/polizeimeldungen/pressemitteilung.103...,Gestern Nachmittag wurde in Mitte bei einem Ve...,0092


In [302]:
import spacy
nlp = spacy.load("de_core_news_lg")


def lemmatize_col(col):
    return ' '.join([token.lemma_ for token in nlp(col) if not token.is_stop and not token.is_punct])
    

In [303]:
df['details'] = df['details'].fillna('')
df['details_lemma'] = df['details'].apply(lemmatize_col)
df.head()

,date,title,location,link,details,number,details_lemma
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841,Zusammenhang Oktober Jahr erfolgt Brandanschla...
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801,Nacht gemeinschaftlich begangen Raub jugendlic...
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316,zwischenzeitlich Landeskriminalamt Rahmen inte...
3,2021-02-04 11:00:00,Gefährliche Körperverletzung,Mitte,/polizei/polizeimeldungen/pressemitteilung.960...,Die Kriminalpolizei der Direktion 2 bittet um ...,0276,Kriminalpolizei Direktion 2 bitten Mithilfe un...
4,2021-01-12 13:02:00,Verkehrsunfall mit schwerverletztem E-Bike-Fahrer,Mitte,/polizei/polizeimeldungen/pressemitteilung.103...,Gestern Nachmittag wurde in Mitte bei einem Ve...,0092,Gestern Nachmittag Mitte Verkehrsunfall E-Bike...


In [304]:
import spacy
from spacy.matcher import Matcher
import pandas as pd

nlp = spacy.load("de_core_news_lg")
matcher = Matcher(nlp.vocab)

# 🔹 Mehr-Wort-Phrasen explizit aufnehmen
crime_categories = {
    "diebstahl": [["stehlen"], ["entwenden"], ["einbrechen"], ['Einbrecher'], ["Taschendiebstahl"], ['Taschendieb'], ["Raub"], ['rauben'],
                   ["Einbrecher"], ["Raubüberfall"], ["Autodiebstahl"], ["Wertgegenstände"], ['klauen'],
                   ["Ladendiebstahl"], ["Fahrraddiebstahl"], ["Kabeldiebstahl"], ["Metallklau"],
                   ["Einbruchdiebstahl"], ["Kfz-Diebstahl"], ["Bargelddiebstahl"], ['Diebstahl']],
    
    "gewaltverbrechen": [["angreifen"], ['Angreifer'], ["Niederschlagen"], ["Mord"], ["Totschlag"], ["schlagen"],
                          ["Messerattacke"], ['Messer'],["Körperverletzung"], ["Schläge"], ["Bewaffnet"], ["Schusswaffe"],
                          ["Schlägerei"], ["Gewalt"], ["Überfall"], ['überfallen'],
                          ["Angriff"], ["Stichverletzung"], ["Erpressung"], ['erpressen'], ["Faustschlag"],
                          ["Kidnapping"], ["Geiselnahme"], ["Messerattacke"],["Amoklauf"], ['schießen']],
    
    "drogen": [["Droge"], ['Drogen'], ["Cannabis"], ["Kokain"], ["Dealer"], ["Rauschgift"], ["schmuggeln"], ["Heroin"], ['dealen'],
                ["Speed"], ["Betäubungsmittel"], ["Ecstasy"], ["Crack"], ["Meth"], ["Drogenhandel"],
                ["Suchtmittel"], ["Opiaten"], ["Drogenmissbrauch"], ["Drogenschmuggel"]],
    
    "betrug": [["betrügen"], ["erschleichen"], ["Kreditkartenbetrug"], ["Fake-Shop"],['Betrug'],
               ["Identitätsdiebstahl"], ["Täuschung"], ['täuschen'], ["Geldfälschung"], ["Internetbetrug"],['Falschgeld'],
               ["Scam"], ["Finanzbetrug"], ["Versicherungsbetrug"], ["Aktienbetrug"], ["Betrügerische", "Masche"],
               ["Geldwäsche"], ["Korruption"], ["Insiderhandel"], ["veruntreuen"],
                ["Steuerhinterziehung"], ['Steuern', 'hinterziehen'], ["Subventionsbetrug"], ["Bilanzfälschung"]],
    
    "vandalismus": [["beschädigen"], ['Beschädigung'], ["zerstören"], ["einschlagen"], ["randalieren"], ["brandstiften"],
                     ["Grafitti"], ["einschlagen"], ['zerkratzen'], ["Sachschaden"], ["Brandanschlag"],
                     ["Anzünden"], ["Brandstiftung"], ["Explosion"], ["Feuer",  "legen"],
                    ["Mülltonnenbrand"]],
    
    "hasskriminalität": [["fremdenfeindlich"], ["beleidigen"], ["Rassismus"], ["antisemitisch"], ["diskriminieren"],
                          ["transfeindlich"], ["Beleidigung"], ["Fremdenfeindlichkeit"], ["Diskriminierung"],
                          ["Hassverbrechen"], ["Antisemitismus"], ["Homophobie"], ["queerfeindlich"],
                          ["Mobbing"],['mobben'],['hetzen'], ["Hetze"], ["Verfassungsfeindlich"]],
    
    "sexualdelikte": [["Vergewaltigung"],['vergewaltigen'], ["sexuell", "belästigen"], ["Übergriff"], ["Exhibitionist"],
                       ["Nötigung"],['nötigen'], ["Missbrauch"], ["missbrauchen"], ["sexuelle", "Belästigung"],
                       ["sexueller", "Übergriff"], ["Pädophilie"], ["Kinderpornografie"], ["sexuelle", "Nötigung"]],
    
    "verkehrsdelikte": [["Führerschein", "entziehen"],['Verkehrsunfall'], ["Unfallflucht"], ["Alkohol", "Steuer"],
                         ["Geschwindigkeit"], ["Rote", "Ampel", "überfahren"], ["falsches", "Parken"],
                         ["Fahrerflucht"], ["Handy","Steuer"], ['angefahren'], ['überfahren']],
    
    "sonstige": []
}


# 🔥 `LEMMA`-basiertes Matching
for category, keyword_lists in crime_categories.items():
    patterns = [[{"LEMMA": word} for word in words] for words in keyword_lists]
    matcher.add(category, patterns)

# 💡 Funktion zur Klassifikation
def classify_crime(text):
	doc = nlp(text)  
	matches = matcher(doc)

	if matches:
		return ', '.join(set([nlp.vocab.strings[match[0]] for match in matches]))

	return "sonstige"


In [305]:
df['category'] = df['details_lemma'].apply(classify_crime)

In [306]:
df.head()

,date,title,location,link,details,number,details_lemma,category
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841,Zusammenhang Oktober Jahr erfolgt Brandanschla...,vandalismus
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801,Nacht gemeinschaftlich begangen Raub jugendlic...,"gewaltverbrechen, diebstahl"
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316,zwischenzeitlich Landeskriminalamt Rahmen inte...,betrug
3,2021-02-04 11:00:00,Gefährliche Körperverletzung,Mitte,/polizei/polizeimeldungen/pressemitteilung.960...,Die Kriminalpolizei der Direktion 2 bittet um ...,0276,Kriminalpolizei Direktion 2 bitten Mithilfe un...,gewaltverbrechen
4,2021-01-12 13:02:00,Verkehrsunfall mit schwerverletztem E-Bike-Fahrer,Mitte,/polizei/polizeimeldungen/pressemitteilung.103...,Gestern Nachmittag wurde in Mitte bei einem Ve...,0092,Gestern Nachmittag Mitte Verkehrsunfall E-Bike...,verkehrsdelikte


In [307]:
df['category'].value_counts()

category
sonstige                                        3486
verkehrsdelikte                                 2424
gewaltverbrechen                                1899
vandalismus                                     1207
diebstahl                                        751
                                                ... 
hasskriminalität, drogen, vandalismus              1
hasskriminalität, diebstahl, vandalismus           1
hasskriminalität, vandalismus, diebstahl           1
drogen, vandalismus, diebstahl                     1
sexualdelikte, gewaltverbrechen, vandalismus       1
Name: count, Length: 83, dtype: int64

In [309]:
df['title'] = df['title'].fillna('')
df.loc[df['category'] == 'sonstige', 'category'] = df.loc[df['category'] == 'sonstige', 'title'].apply(classify_crime)


In [295]:
df['category'].value_counts()

cat
sonstige                                                            2761
verkehrsdelikte                                                     2535
gewaltverbrechen                                                    2026
vandalismus                                                         1502
diebstahl                                                            859
                                                                    ... 
drogen, gewaltverbrechen, hasskriminalität, verkehrsdelikte            1
gewaltverbrechen, vandalismus, hasskriminalität, verkehrsdelikte       1
hasskriminalität, vandalismus, diebstahl                               1
drogen, vandalismus, diebstahl                                         1
sexualdelikte, gewaltverbrechen, vandalismus                           1
Name: count, Length: 82, dtype: int64

In [310]:
all_categories = set([category for sublist in df['category'].str.split(', ') for category in sublist])

for category in all_categories:
    df[category] = df['category'].apply(lambda x: category in x.split(', '))

In [311]:
df.head()

,date,title,location,link,details,number,details_lemma,category,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,sonstige,hasskriminalität,gewaltverbrechen
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841,Zusammenhang Oktober Jahr erfolgt Brandanschla...,vandalismus,False,True,False,False,False,False,False,False,False
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801,Nacht gemeinschaftlich begangen Raub jugendlic...,"gewaltverbrechen, diebstahl",False,False,False,False,False,True,False,False,True
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316,zwischenzeitlich Landeskriminalamt Rahmen inte...,betrug,False,False,False,False,True,False,False,False,False
3,2021-02-04 11:00:00,Gefährliche Körperverletzung,Mitte,/polizei/polizeimeldungen/pressemitteilung.960...,Die Kriminalpolizei der Direktion 2 bittet um ...,0276,Kriminalpolizei Direktion 2 bitten Mithilfe un...,gewaltverbrechen,False,False,False,False,False,False,False,False,True
4,2021-01-12 13:02:00,Verkehrsunfall mit schwerverletztem E-Bike-Fahrer,Mitte,/polizei/polizeimeldungen/pressemitteilung.103...,Gestern Nachmittag wurde in Mitte bei einem Ve...,0092,Gestern Nachmittag Mitte Verkehrsunfall E-Bike...,verkehrsdelikte,True,False,False,False,False,False,False,False,False


In [314]:
df.loc[df['category'] == 'sonstige', ['title', 'details_lemma']]

,title,details_lemma
8,Polizei Berlin zieht Bilanz zur Versammlungsla...,überwiegend störungsfrei verlaufen gestrig Ver...
14,E-Scooter in Flammen,Brandkommissariat Landeskriminalamt Ermittlung...
16,Fahrzeuge in Brand gesetzt,Brandkommissariat Landeskriminalamt Ermittlung...
18,Farbschmierereien an mehreren Hausfassaden,Streife Polizeiabschnitt 55 stellen gestern Ab...
19,Einsatzlagen zum Jahreswechsel,Polizei Berlin bereiten aktuell intensiv Einsa...
...,...,...
277,Vermisste Seniorin wieder da,Suche vermisst Seniorin Pankow beenden Frau Re...
283,Silvesternacht – Haftbefehle erlassen,Zusammenhang Silvesternacht Ermittlungsrichter...
291,Bilanz zum Jahreswechsel 2024/2025,Großeinsatz Jahreswechsel stellen Polizei Berl...
295,Versammlungen mit Bezug zum Nahostkonflikt am ...,Mittag Nachmittag Silvestertag Polizei Berlin ...


In [312]:
df.to_csv('oh_encoded_categories.csv')